In [1]:
!pip install unidecode

In [2]:
from google.colab import drive
import os

# Grab the paths to the metadata files and the waves file from google drive
drive.mount("/content/drive")
BASE_DIR = "/content/drive/My Drive/Computer Science/Jabbel/data"
csv_path = os.path.join(BASE_DIR, "metadata.csv")
audio_dir = os.path.join(BASE_DIR, "wavs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset
```python
datasets.load_dataset()
```
This function is from HuggingFace that loads in audio dir based on the file names e.g (metedata.csv and wavs dir)

In [3]:
import datasets

dataset = datasets.load_dataset(
    "csv",
    data_files = csv_path,
    # Tells HF the divider that seperates the ID : Transcript... (normally a , but it is a | for me) 
    sep = "|",                                      
     # Tells PyTorch the headers ID : Transcript : Norm_Trans for each record in the csv, as there's no headers
    column_names=["file_id", "transcription", "normalized_transcription"],   
    split = "train"                 
)

# Do not need the normal one as the normalized one will be used for training
dataset = dataset.remove_columns(["transcription"])

# Create Full Paths to .wav files
Because the csv file only lists the ID and not the file extention, I need to map over the csv and manually add them so the ful path is built
1. <code>batch</code>: This takes mutiple rows at a time, instead of looking at one row at a time.
2. <code>f"{fid}.wav"</code>: This takes the ID and ads .wav onto the end of each ID
3. <code>batch["audio"]</code>: This adds a new column called audio with all the correct path names
4. <code>dataset.map</code>: This tells the dataset to run the function across every single row in the dataset


In [4]:
def add_audio_paths(batch):
    batch["audio"] = [os.path.join(audio_dir, f"{fid}.wav") for fid in batch["file_id"]]
    return batch

dataset = dataset.map(add_audio_paths, batched = True)

# Cast The Path Strings To Actual Audio Files
1. Makes the audio column, currently just text filenames, be treatd as actual audio rather than just plane text
2. <code>Audio(sampling_rate = 16_000)</code>: Tells the model no matter what make alll these files 16,000 Hz to make sure they are all the same Hz

In [5]:
dataset = dataset.cast_column("audio", datasets.Audio(sampling_rate = 16_000))

print(dataset[0]["audio"])
print(dataset[0])

{'file_id': 'LJ001-0001', 'normalized_transcription': 'Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7ab341148770>}


# Cleaning Dataset of Blank Transcriptions and Accent Chars
In the readme file it says that 19 files have special non ASCHII chars, therefore i need to clean them and just turn them into non accent chars just so the tokenization works

In [6]:
# Find None and Empty Strings in the Dataset
all_texts = dataset["normalized_transcription"]

has_none = None in all_texts
has_empty = "" in all_texts

print(f"Contains None: {has_none}")
print(f"Contains Empty Strings: {has_empty}")

Contains None: True
Contains Empty Strings: False


In [8]:
from unidecode import unidecode 

def clean_text(batch):
    # if 't' is None, return an empty string "" 
    batch["normalized_transcription"] = [
        unidecode(t) if t is not None else "" 
        for t in batch["normalized_transcription"]
    ]
    return batch

dataset = dataset.map(clean_text, batched=True)

Map:   0%|          | 0/13100 [00:00<?, ? examples/s]

# Creating Tokenization Function

In [9]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders

TOKENIZER_PATH = os.path.join(BASE_DIR, "tokenizer.json")

def get_tokenizer(save_path = TOKENIZER_PATH):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.add_special_tokens(["[PAD]", "[UNK]", " "])
    tokenizer.add_tokens(list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz' "))

    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space = True)
    tokenizer.decoder = decoders.ByteLevel()
    # tokenizer.save(save_path)
    return tokenizer


In [10]:
tokenizer = get_tokenizer()
tokenizer.get_vocab()

{'o': 43,
 'D': 6,
 'i': 37,
 's': 47,
 'I': 11,
 'f': 34,
 'Z': 28,
 'u': 49,
 ' ': 2,
 'N': 16,
 't': 48,
 '[PAD]': 0,
 'M': 15,
 'a': 29,
 'z': 54,
 'X': 26,
 'U': 23,
 'm': 41,
 'b': 30,
 'R': 20,
 'P': 18,
 'p': 44,
 'E': 7,
 'n': 42,
 'e': 33,
 'S': 21,
 'G': 9,
 'g': 35,
 'B': 4,
 'd': 32,
 "'": 55,
 'y': 53,
 'V': 24,
 'k': 39,
 'j': 38,
 'O': 17,
 'F': 8,
 'r': 46,
 'l': 40,
 'A': 3,
 'Q': 19,
 'w': 51,
 'T': 22,
 'c': 31,
 '[UNK]': 1,
 'x': 52,
 'K': 13,
 'C': 5,
 'J': 12,
 'v': 50,
 'h': 36,
 'L': 14,
 'H': 10,
 'q': 45,
 'W': 25,
 'Y': 27}

# Creating Wrapper For Dataset
This is where i create the brdige between the Hugging Face dataset and the tokenizer objects and how PyTorch can understand the dataset 

In [11]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# The Wrapper Class
class WrapDataset(Dataset):
    def __init__(self, dataset, tokenizer = None, num_samples = None):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.num_samples = len(dataset) if num_samples is None else num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Convert the waveform to a torch tensor
        waveform = torch.from_numpy(item["audio"]["array"]).float()

        text = item["normalized_transcription"]

        if self.tokenizer:
            # Tokenize the text
            encoded = self.tokenizer.encode(text)
            return {"audio": waveform,"text": text ,"input_ids": encoded.ids}
        
        return {"audio": waveform, "text": text}


In [12]:
# The Padding Function 
def collate_fn(batch):
    # Find the longest audio file in current batch
    max_audio_len = max([item["audio"].shape[0] for item in batch])

    # Find the longest text setence in current batch
    has_input_ids = "input_ids" in batch[0]
    max_ids_len = 0
    if has_input_ids:
        max_ids_len = max([len(item["input_ids"]) for item in batch])

    # Pad the audio sequences with 0s at the end so they all match max_audio_len
    audio_tensor = torch.stack([
        F.pad(item["audio"], (0, max_audio_len - item["audio"].shape[0]))
        for item in batch
    ])

    output_dict = {
        "audio": audio_tensor,
        "text": [item["text"] for item in batch]
    }

    # Pad the text IDs so they all match max_ids_len
    if has_input_ids:
        input_ids_tensor = torch.stack([
            F.pad(
                torch.tensor(item["input_ids"]),
                (0, max_ids_len - len(item["input_ids"])),
                value=0 
            )
            for item in batch
        ])
        output_dict["input_ids"] = input_ids_tensor

    return output_dict

In [13]:
# The Manager Fuction 
def get_dataloader(target_dataset, tokenizer, batch_size, num_examples = None):
    # Wrap the Hugging face dataset in the PyTorch Class
    dataset = WrapDataset(target_dataset, tokenizer = tokenizer, num_samples = num_examples)

    # Create the PyTorch DataLoader
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = True,
        collate_fn = collate_fn,
        num_workers = 2
    )
    return dataloader

In [14]:
# Create the DataLoader
train_loader = get_dataloader(dataset, tokenizer, batch_size = 64)

# Grab the first batch
first_batch = next(iter(train_loader))

print(f"Audio Tensor Shape: {first_batch['audio'].shape}")
print(f"Text IDs Tesnsor Shape: {first_batch['input_ids'].shape}")

Audio Tensor Shape: torch.Size([64, 160611])
Text IDs Tesnsor Shape: torch.Size([64, 158])


# Network Architecture
For this architecture we will be using a Convolutional Neural Network (CNN) to process the audio data and a Recurrent Neural Network (RNN) to generate the text.

In [30]:
import torch.nn as nn 

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, kernel_size = 4):
        super().__init__()
        
        # Feature Extraction (Change channels, keep length the same)
        self.conv1 = nn.Conv1d(
            in_channels, 
            out_channels, 
            kernel_size = kernel_size, 
            padding = "same"
            )

        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()

        # If the channel sizes are differnet, a 1x1 convolution  to match the channel sizes
        if in_channels != out_channels:
            self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size = 1)
        else:
            self.shortcut = nn.Identity()

        # Downsample (Compress the sequence length) 
        # Note: Adding extra padding to stop the kernal chopping off the ends of the sequecne during stride
        self.conv2 = nn.Conv1d(
            out_channels, 
            out_channels, 
            kernel_size = kernel_size, 
            stride = stride, 
            padding = (kernel_size // 2) // 2 # // 2 because in test it was adding a +1 when it shouldn't of 
            )


    # Functions to pass the data through the network
    def forward(self, x):
        output = self.conv1(x)
        output = self.bn1(output)
        output = self.relu(output)
        output = output + self.shortcut(x)
        output = self.conv2(output)
        return output
       
        

In [31]:
# Testing with dummy data

# Batch Size 32, 1 Input Channel (mono audio), 16000 samples
dummy_audio = torch.randn(32, 1, 16000) 
res_block = ResidualBlock(in_channels = 1, out_channels = 16, stride = 2)
test_output = res_block(dummy_audio)

print(f"Original Shape: {dummy_audio.shape}")
print(f"Output Shape:   {test_output.shape}")

Original Shape: torch.Size([32, 1, 16000])
Output Shape:   torch.Size([32, 16, 8000])


In [32]:
class DownsamplingNetwork(nn.Module):
    def __init__(
        self,
        embedding_dim = 128,
        hidden_dim = 64,
        in_channels = 1,
        initial_mean_pooling_kernel_size = 2,
        strides = [6, 6, 8, 4, 2]
    ):
        super().__init__()
        self.layers = nn.ModuleList()

        # The Initial Cut
        self.mean_pooling = nn.MaxPool1d(kernel_size = initial_mean_pooling_kernel_size)
        
        # The Loop (Building the Blocks)
        for i in range(len(strides)):
            self.layers.append(
                ResidualBlock(
                    in_channels = hidden_dim if i > 0 else in_channels,
                    out_channels = hidden_dim,
                    stride = strides[i],
                    kernel_size = 8
                )
            )
            
        # Expands to the embedding dimension
        self.final_conv = nn.Conv1d(
            hidden_dim, 
            embedding_dim, 
            kernel_size = 4, 
            padding = "same"
        )

    def forward(self, x):
        # Apply the initial pooling
        x = self.mean_pooling(x)
        
        # Pass the data through all 5 residual blocks
        for layer in self.layers:
            x = layer(x)
            
        # Apply the final convolution
        x = self.final_conv(x)
        
        return x

In [33]:
# Testing the full endcoder

# Initialize the full audio encoder
audio_encoder = DownsamplingNetwork()

# Pass the audio through the encoder
encoded_audio = audio_encoder(dummy_audio)

print(f"Original Audio Shape: {dummy_audio.shape}")
print(f"Encoded Audio Shape:  {encoded_audio.shape}")

Original Audio Shape: torch.Size([32, 1, 16000])
Encoded Audio Shape:  torch.Size([32, 128, 2])
